In [ ]:

class MyKNNReg:
    def __init__(self, k = 3, metric = 'euclidean', weight = 'uniform'):
        self.k = k 
        self.train_size = None
        self.metric = metric
        self.weight = weight

    def __str__(self):
        return f"MyKNNReg class: k={self.k}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.X = X.copy()
        self.y = y.copy()
        self.train_size = self.X.shape

    def _get_distances(self, x: pd.Series):
        if self.metric == 'euclidean':
            return np.sqrt(((self.X - x) ** 2).sum(axis=1))
        if self.metric == 'chebyshev':
            return (self.X - x).abs().max(axis=1)
        if self.metric == 'manhattan':
            return (self.X - x).abs().sum(axis=1)
        if self.metric == 'cosine':
            return 1 - (self.X * x).sum(axis=1) / ( np.sqrt((self.X ** 2).sum(axis=1)) * ((x ** 2).sum() ** 0.5) )
    
    def predict(self, X: pd.DataFrame):
        y_pred = []
        for index, row in X.iterrows():
            distances = self._get_distances(row)
            distances.sort_values(ascending=True, inplace=True)
            train_indexes = distances.index[:self.k]
            k_nearest_neighbor_distances = distances.loc[train_indexes]
            k_nearest_neighbor_y = self.y.loc[train_indexes]
            if self.weight == 'uniform':
                y_hat = k_nearest_neighbor_y.mean()
            if self.weight == 'rank':
                k_nearest_neighbor_rank = pd.Series(range(1, self.k + 1), train_indexes)
                sum_weight = (1 / k_nearest_neighbor_rank).sum()
                y_hat = (k_nearest_neighbor_y / k_nearest_neighbor_rank).sum() / sum_weight
            if self.weight == 'distance':
                sum_weight = (1 / k_nearest_neighbor_distances).sum()
                y_hat = (k_nearest_neighbor_y / k_nearest_neighbor_distances).sum() / sum_weight
            y_pred.append(y_hat)
        return pd.Series(y_pred, X.index)
        
        
                
            
        